# WHaLE Example Usage and functionality

This notebook is purely for demonstration purposes on how to configure and use WHaLE. The first note is that a user's `library_path` should have three subfolders: "ORBIT", "WOMBAT", and "FLORIS", each setup in a way that aligns with the respective software's library specifications.

## Imports and environment set up

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from floris.tools.visualization import visualize_cut_plane
from floris.tools.wind_rose import WindRose

from whale import Project

pd.options.display.float_format = '{:,.4f}'.format

## Determine the configurations and create a Project

**NOTE**: Make sure in the WOMBAT config files that the library path is the same that gets printed in this first code block when you run the notebook

In [2]:
library_path = Path("library").resolve()
orbit_config = "Ocean_Wind1.yaml"
wombat_config = "Ocean_Wind_1_100pct_reduction-90.yaml"
floris_config = "Ocean_Wind_1_gch.yaml"
print(library_path)

/Users/rhammond/Documents/GitHub/WHaLE/examples/library


In [3]:
project = Project(
    library_path=library_path,
    weather=library_path / "weather" / "alpha_ventus_weather_2002_2014.csv",
    orbit_config=orbit_config,
    wombat_config=wombat_config,
    floris_config=floris_config,
)

ORBIT library intialized at '/Users/rhammond/Documents/GitHub/WHaLE/examples/library'


In [4]:
# show that everything is initialized
print(f"ORBIT Capacity: {project.orbit.capacity} MW")
print(f"WOMBAT Capacity: {project.wombat.windfarm.capacity / 1000:.0f} MW")
print(f"FLORIS coordinates: {list(zip(project.floris.layout_x, project.floris.layout_y))}")

ORBIT Capacity: 1104 MW
WOMBAT Capacity: 1380 MW
FLORIS coordinates: [(0.0, 0.0), (1041.0, -1059.0), (2082.0, -2119.0), (3122.0, -3179.0), (4163.0, -4238.0), (5204.0, -5298.0), (6244.0, -6358.0), (7285.0, -7417.0), (-1455.0, -1241.0), (495.0, -3473.0), (1470.0, -4589.0), (2444.0, -5705.0), (3419.0, -6821.0), (4394.0, -7936.0), (5369.0, -9052.0), (-2872.0, -2451.0), (-1910.0, -3577.0), (-947.0, -4703.0), (15.0, -5830.0), (978.0, -6956.0), (1940.0, -8082.0), (2903.0, -9209.0), (3866.0, -10335.0), (-4386.0, -3742.0), (-2459.0, -5993.0), (-1495.0, -7119.0), (-532.0, -8244.0), (431.0, -9370.0), (1395.0, -10496.0), (2358.0, -11621.0), (3321.0, -12747.0), (-5975.0, -5098.0), (-5013.0, -6225.0), (-4052.0, -7352.0), (-3090.0, -8479.0), (-2128.0, -9606.0), (-1167.0, -10733.0), (-205.0, -11861.0), (757.0, -12988.0), (1718.0, -14115.0), (-8346.0, -5173.0), (-7384.0, -6300.0), (-6422.0, -7427.0), (-5461.0, -8554.0), (-4499.0, -9681.0), (-3537.0, -10808.0), (-2576.0, -11936.0), (-1614.0, -13063.0), 

## Run the analyses and calculate results

This separately calls the `run` methods for each of the ORBIT `ProjectManager` and WOMBAT `Simualation`, in that order. Alternatively, these could just be called on their own, just like we must with FLORIS for the time being.

**NOTE: In reality, it will need to be determined how we should actually run FLORIS to get the outputs that we want, so this usage is a crude placeholder**

In [5]:
project.run_capex_opex()

project.floris.calculate_wake()

In [6]:
project.orbit.capex_breakdown

{'Array System': 67608092.08505006,
 'Export System': 431889698.00168,
 'Offshore Substation': 171240450.0,
 'Scour Protection': 18241760,
 'Substructure': 514120861.30114037,
 'Array System Installation': 27110772.72011773,
 'Export System Installation': 213972542.53604308,
 'Offshore Substation Installation': 8769728.444629062,
 'Scour Protection Installation': 63441940.63926946,
 'Substructure Installation': 79534112.74514198,
 'Turbine Installation': 91199080.18104854,
 'Turbine': 1656000000,
 'Soft': 599648640.0,
 'Project': 151250000.0}

In [7]:
project.wombat.metrics.time_based_availability("project", "windfarm")

,windfarm
0,0.3318


In [8]:
# OpEx, in millions, annually
project.wombat.metrics.opex(frequency="project") / 1e6

The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.

,OpEx
0,207.2722


In [9]:
# TODO: Calculate the wind rose from the WOMBAT weather profile
# TODO: Find the ERA5 data that has the u/v components and compute the direction
# TODO: Use this to then capture the AEP per the FLORIS workflow

# TODO: Once workflow is finalized, bring into a new project.run() method

# wind_rose = WindRose()
# ws = project.wombat.weather.windspeed.values
# wd = project.wombat.weather.wind_direction.values
# wr_df = wind_rose.make_wind_rose_from_user_data(wd, ws)
# wind_rose.plot_wind_rose();

# aep_wake = project.floris.get_farm_AEP_wind_rose_class(
#     wind_rose=wind_rose,
#     cut_in_wind_speed=3.0,  # Wakes are not evaluated below this wind speed
#     cut_out_wind_speed=25.0,  # Wakes are not evaluated above this wind speed
# )
# print(f"Farm AEP (with cut_in/out specified): {aep_wake / 1.0e9:.3f} GWh")

# aep_no_wake = project.floris.get_farm_AEP_wind_rose_class(
#     wind_rose=wind_rose, no_wake=True
# )
# print(f"Farm AEP (no_wake=True): {aep_no_wake / 1.0e9:.3f} GWh")

In [10]:
project.wombat.env.cleanup_log_files()